In [ ]:
!mkdir /kaggle/temp

In [ ]:
!wget -O /kaggle/temp/Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-$(uname)-$(uname -m).sh"

In [ ]:
!rm -rf /kaggle/temp/conda
!bash /kaggle/temp/Miniforge3.sh -b -p /kaggle/temp/conda

In [ ]:
import subprocess
subprocess.run(['/kaggle/temp/conda/bin/mamba', 'install', '-y', 'python=3.13', 'cuda-version=13.0', 'pytorch=2.11',
                'pillow', 'matplotlib', 'scipy', 'nvimgcodec', 'nccl', 'cuda-cudart-dev'], check=True)
subprocess.run(['/kaggle/temp/conda/bin/python', '-c',
                'import sys, torch, triton; print(sys.version); print(torch.__version__, torch.version.cuda, torch.cuda.nccl.version(), triton.__version__)'],
               check=True)

In [ ]:
!rm -rf /kaggle/temp/music_deep
!git clone -b exp-038-phase-b-joint-refiner https://github.com/kwon-young/music_deep.git /kaggle/temp/music_deep
!git -C /kaggle/temp/music_deep log --oneline -1

In [ ]:
import os

# Run settings. Test run: 10 epochs; full run: EPOCHS = 100, EXP_DIR = 'experiments/038_phase_b_joint_refiner',
# TIMEOUT = '11h'.
EPOCHS = 10
EXP_DIR = 'experiments/038_phase_b_test_10ep'
TIMEOUT = '3h'

# The COCO JSON is parsed fresh (~1 min) with the current code.
os.makedirs('/kaggle/temp/cache', exist_ok=True)

# Exp 037 detector + its Phase A refiner, from the private dataset kwonyoungchoi/music-deep-038-init
# (a dataset rather than a model: `kaggle kernels push` keeps dataset sources but drops model sources).
print(sorted(os.listdir('/kaggle/input')))
wanted = {'init_037.pt': '/kaggle/temp/init_037.pt', 'refiner_037_phase_a.pt': '/kaggle/temp/refiner_037_phase_a.pt'}
found = {name: [] for name in wanted}
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if d not in ('ssl-imslp', 'trompa-coco', 'trompa-coco-cache')]
    for name in files:
        if name in wanted:
            found[name].append(os.path.join(root, name))
print(found)
for name, paths in found.items():
    assert len(paths) == 1, (name, paths)
    if os.path.lexists(wanted[name]):
        os.remove(wanted[name])
    os.symlink(paths[0], wanted[name])

In [ ]:
# Exp 038: Exp 037 continued on all pages with the line endpoint refiner trained inside the detector
# step (Exp 035 Phase B). Checkpoints every epoch: if `timeout` stops the run early, the last one is
# kept (for 100 epochs, the LR is already < 3e-6 after epoch 90).
!PYTHONPATH=/kaggle/temp/music_deep /kaggle/temp/conda/bin/mamba run timeout {TIMEOUT} torchrun --nproc_per_node=2 /kaggle/temp/music_deep/src/train_detection.py \
    --exp_dir {EXP_DIR} \
    --patch_size 64 \
    --epochs {EPOCHS} \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --headless \
    --cache_dir /kaggle/temp/cache/ \
    --use_sdpa \
    --log_epoch_interval 1 \
    --warmup_epochs 1 \
    --lr 1e-4 \
    --compile \
    --extract_patch_min 54 \
    --extract_patch_max 74 \
    --class_weight_variance_lambda 0.5 \
    --class_weight_size_difficulty_max 8.0 \
    --detector_checkpoint /kaggle/temp/init_037.pt \
    --line_refiner \
    --refiner_init /kaggle/temp/refiner_037_phase_a.pt \
    --refiner_lr_scale 1.0

In [ ]:
# All 351 pages with the full line pipeline (refine -> NMS 4 px -> noisy-OR); the refiner is read
# from the same checkpoint. Evaluate locally (pycocotools is not installed here).
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run python /kaggle/temp/music_deep/src/inference_coco.py \
    --checkpoint {EXP_DIR}/train_detection/checkpoints/latest_model.pt \
    --refiner_checkpoint {EXP_DIR}/train_detection/checkpoints/latest_model.pt \
    --output_dir {EXP_DIR}/inference_full_pipeline \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa

In [ ]:
# Coarse head only (the old top-k path), to separate head and refiner changes.
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run python /kaggle/temp/music_deep/src/inference_coco.py \
    --checkpoint {EXP_DIR}/train_detection/checkpoints/latest_model.pt \
    --output_dir {EXP_DIR}/inference \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa